In [1]:
from src.common.constants import Constants as consts
from src.models.model_trainer import ModelTrainer
from src.models.logistic_regression_clf import LogisticRegressionClf, objective_acc
from src.common.constants import ExperimentPreprocessConfig
from src.preprocessing.experiment_preprocessor import ExperimentPreprocessor


FRACTION = 0.5
USE_AUDIO_ID_SAMPLING = False
FILE_SUFFIX = consts.wavlm_emb_suffix
N_TRIALS = 20
MODEL = LogisticRegressionClf
OBJECTIVE = objective_acc

print(f"""
    FRACTION: {FRACTION}
    USE_AUDIO_ID_SAMPLING: {USE_AUDIO_ID_SAMPLING}
    FILE_SUFFIX: {FILE_SUFFIX}
    N_TRIALS: {N_TRIALS}
    MODEL: {MODEL.__name__}
    OBJECTIVE: {OBJECTIVE.__name__}
    """)


    FRACTION: 0.5
    USE_AUDIO_ID_SAMPLING: False
    FILE_SUFFIX: _wavlm
    N_TRIALS: 20
    MODEL: LogisticRegressionClf
    OBJECTIVE: objective_acc
    


In [ ]:
exp_preprocessor = ExperimentPreprocessor(
    load_file_name=consts.feature_extracted,
    save_file_name=consts.feature_extracted,
    feat_suffix=FILE_SUFFIX
)

balance_strategy_ratios = {
    "unbalanced": [None],
    "oversample": [0.5, 0.75, 1.0],
    "undersample": [0.5, 0.75, 1.0],
    "mix": [[0.5, 0.75], [0.5, 1.0], [0.75, 1.0]],
}

trainer = ModelTrainer()

In [ ]:
from sklearn.preprocessing import StandardScaler
from src.common.logger import setup_logger
logger = setup_logger("train_all_balancers", log_to_console=True)

def create_comparision_dict(balance_strategy_ratios):
    comparison_dict = {}
    for strategy, ratios in balance_strategy_ratios.items():
        comparison_dict[f"{strategy}:{ratios}"] = ratios
    return comparison_dict

def train_all_balancers(
    model,
    objective,
    n_trials: int,
    exp_preprocessor: ExperimentPreprocessor,
    balance_strategy_ratios: dict[str, list[float] | list[list[float]]],
):
    comparison_dict = create_comparision_dict(balance_strategy_ratios)

    dev_set_preprocessing_config = ExperimentPreprocessConfig(
        splits_names=["dev"],
        fraction=FRACTION,
        use_audio_id_sampling=USE_AUDIO_ID_SAMPLING,
        balance_splits_strategy=None,
        use_standardize=False,
    )
    data_for_exp = exp_preprocessor.preprocess_data(**dev_set_preprocessing_config.get_dict())
    print(data_for_exp)
    X_dev, y_dev = exp_preprocessor.get_X_y(metadata=data_for_exp["dev"][0], features=data_for_exp["dev"][1])
    X_dev = StandardScaler().fit_transform(X_dev)
    logger.info("Dev shape:", X_dev.shape, y_dev.shape)

    for strategy, ratios in balance_strategy_ratios.items():
        for ratio in ratios:
            logger.info(f"Training with strategy: {strategy}, ratio: {ratio}")
            preprocessing_config = ExperimentPreprocessConfig(
                splits_names=["train"],
                fraction=FRACTION,
                use_audio_id_sampling=USE_AUDIO_ID_SAMPLING,
                balance_splits_strategy=[strategy, ratio],
                use_standardize=False,
            )

            data_for_exp = exp_preprocessor.preprocess_data(**preprocessing_config.get_dict())
            X_train, y_train = exp_preprocessor.get_X_y(
                metadata=data_for_exp["train"][0], features=data_for_exp["train"][1]
            )
            X_train = StandardScaler().fit_transform(X_train)
            logger.info("Train shape:", X_train.shape, y_train.shape)

            trainer.optuna_train(
                model=model,
                objective=objective,
                n_trials=n_trials,
                # max_iter=100,
                X_train=X_train,
                y_train=y_train,
                X_dev=X_dev,
                y_dev=y_dev,
            )

            best_value = trainer.get_best_value()
            comparison_dict_key = f"{strategy}:{ratio}"
            comparison_dict[comparison_dict_key] = best_value

    return comparison_dict


results = train_all_balancers(
    model=MODEL,
    objective=OBJECTIVE,
    n_trials=N_TRIALS,
    exp_preprocessor=exp_preprocessor,
    balance_strategy_ratios=balance_strategy_ratios,
)
print(results)